# Notebook 13: Kingfisher prompt-conditioned scale decomposition on CSD100

This notebook reuses the lightweight Infinity-2B GGUF setup and the same Kingfisher-style implementation from Notebook 12, but runs on 10 deterministic pairs from `csd100`.

Each CSD100 item is stored as `<object>+<style>/00.jpg`. The notebook parses both labels, decomposes each image with its own object prompt family, composes the content source with the style reference, and generates 10 combined images.


In [ ]:
from pathlib import Path
import gc
import importlib.util
import os
import re
import shutil
import subprocess
import sys
import time

# All files are kept under one directory so the notebook can be rerun safely.
ROOT = Path('/content/notebook_13_kingfisher_csd100')
PORT_DIR = ROOT / 'gguf_port'
OFFICIAL_DIR = PORT_DIR / 'Infinity'
ASSET_DIR = ROOT / 'assets'
OUTPUT_DIR = ROOT / 'outputs'
for path in (PORT_DIR, ASSET_DIR, OUTPUT_DIR):
    path.mkdir(parents=True, exist_ok=True)

OFFICIAL_REPO = 'https://github.com/FoundationVision/Infinity.git'
GGUF_REPO = 'kzopp/Infinity-2B-GGUF_UNOFFICIAL'
MODEL_PN = '0.25M'  # Colab-friendly 512px adaptation; the paper reports 1M/1024px.
CFG_SCALE = 1.0
TAU = 0.1
SEED = 42
T5_DEVICE = 'cuda'  # T4/Colab: keep T5 off host RAM; use 'cpu' only with a high-RAM runtime.
print('ROOT:', ROOT)
print('MODEL_PN:', MODEL_PN, '| CFG:', CFG_SCALE, '| TAU:', TAU, '| SEED:', SEED, '| T5:', T5_DEVICE)

In [ ]:
# Check the runtime before installing anything.
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name)
    print('VRAM GiB:', round(props.total_memory / 2**30, 2))
else:
    print('WARNING: no GPU detected; inference will be extremely slow.')

In [ ]:
# Install the packages used by the GGUF loader.
# We intentionally do not install torch or flash-attn here: Colab already ships torch,
# and the GGUF loader falls back to PyTorch SDPA when flash-attn is unavailable.
packages = [
    'gguf', 'gradio', 'transformers', 'sentencepiece',
    'easydict', 'typed-argument-parser', 'seaborn', 'kornia',
    'gputil', 'colorama', 'omegaconf', 'timm==0.9.6',
    'decord', 'pytz', 'imageio', 'einops', 'opencv-python', 'accelerate',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)
print('Dependency installation finished.')

In [ ]:
# Clone the official Python architecture only once.
if not OFFICIAL_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', OFFICIAL_REPO, str(OFFICIAL_DIR)], check=True)
else:
    print('Official Infinity source already exists:', OFFICIAL_DIR)

# Download only the files needed from the unofficial GGUF repository.
from huggingface_hub import hf_hub_download

def download_hf_file(filename, target_dir):
    target_dir.mkdir(parents=True, exist_ok=True)
    return Path(hf_hub_download(
        repo_id=GGUF_REPO,
        filename=filename,
        local_dir=str(target_dir),
    ))

PORT_SCRIPT = download_hf_file('generate_image_2b_q8_gguf.py', PORT_DIR)
PORT_UTILS = download_hf_file('infinity_gguf_utils.py', PORT_DIR)
PATCH_DIR = ROOT / 'gguf_patched_source'
PATCHED_BASIC = download_hf_file('Infinity/infinity/models/basic.py', PATCH_DIR)
PATCHED_INFINITY = download_hf_file('Infinity/infinity/models/infinity.py', PATCH_DIR)

# The GGUF repository includes patched source files with optional attention fallbacks.
# Copy them over the matching files in the official source tree.
official_basic = OFFICIAL_DIR / 'infinity' / 'models' / 'basic.py'
official_infinity = OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py'
shutil.copy2(PATCHED_BASIC, official_basic)
shutil.copy2(PATCHED_INFINITY, official_infinity)

# The patched attention module may intentionally expose flash_attn_func=None.
# Guard the official constructor so it selects the PyTorch SDPA fallback safely.
infinity_source = official_infinity.read_text()
old_attention_guard = "customized_kernel_installed = any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
new_attention_guard = "customized_kernel_installed = flash_attn_func is not None and any('Infinity' in arg_name for arg_name in flash_attn_func.__code__.co_varnames)"
if old_attention_guard not in infinity_source:
    raise RuntimeError('Expected optional-attention guard was not found in patched infinity.py')
official_infinity.write_text(infinity_source.replace(old_attention_guard, new_attention_guard, 1))
INFINITY_GGUF = download_hf_file('infinity_2b_reg_Q8_0.gguf', ASSET_DIR)
T5_GGUF = download_hf_file('flan-t5-xl-encoder-Q8_0.gguf', ASSET_DIR)
VAE_PATH = download_hf_file('Infinity/infinity_vae_d32_reg.pth', ASSET_DIR)

print('GGUF model:', INFINITY_GGUF)
print('T5 encoder:', T5_GGUF)
print('VAE:', VAE_PATH)
print('Loader:', PORT_SCRIPT)
print('Loader utility:', PORT_UTILS)

In [ ]:
# Verify the expected files before importing the custom loader.
required_files = [PORT_SCRIPT, PORT_UTILS, PATCHED_BASIC, PATCHED_INFINITY, INFINITY_GGUF, T5_GGUF, VAE_PATH]
missing = [str(path) for path in required_files if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))

for path in required_files:
    print(f'{path.name:40s} {path.stat().st_size / 2**30:.3f} GiB')

assert (OFFICIAL_DIR / 'infinity' / 'models' / 'infinity.py').exists(), 'Official Infinity source is incomplete.'
print('All GGUF, VAE, and official source files are present.')

## Memory-efficient T5 loading

The upstream GGUF script first materializes a complete FP16 state dictionary and then creates a second T5 model. That duplicates most of the encoder in host RAM. The replacement below streams one GGUF tensor at a time into an empty Flan-T5-XL model. It preserves the expected 2048-dimensional text features.

A smaller T5 (such as T5-small/base) is not a drop-in replacement: its hidden width and learned feature space differ from the Infinity-2B cross-attention interface. Using one would require a trained projection or distillation step, so this notebook keeps Flan-T5-XL and reduces peak RAM instead.

In [ ]:
import math
import numpy as np
import gguf

def load_t5_encoder_streaming(gguf_path, device='cpu'):
    from gguf import GGUFReader
    from transformers import T5Config, T5EncoderModel

    key_map = {
        'enc.': 'encoder.',
        '.blk.': '.block.',
        'token_embd': 'shared',
        'output_norm': 'final_layer_norm',
        'attn_q': 'layer.0.SelfAttention.q',
        'attn_k': 'layer.0.SelfAttention.k',
        'attn_v': 'layer.0.SelfAttention.v',
        'attn_o': 'layer.0.SelfAttention.o',
        'attn_norm': 'layer.0.layer_norm',
        'attn_rel_b': 'layer.0.SelfAttention.relative_attention_bias',
        'ffn_up': 'layer.1.DenseReluDense.wi_1',
        'ffn_down': 'layer.1.DenseReluDense.wo',
        'ffn_gate': 'layer.1.DenseReluDense.wi_0',
        'ffn_norm': 'layer.1.layer_norm',
    }

    config = T5Config.from_pretrained('google/flan-t5-xl')
    try:
        from accelerate import init_empty_weights
        with init_empty_weights():
            model = T5EncoderModel(config)
        # Materialize directly as FP16 to avoid allocating a full FP32 T5.
        model = model.to(dtype=torch.float16)
        model.to_empty(device=device)
    except Exception as exc:
        raise RuntimeError(
            'Streaming T5 loading requires the accelerate package and empty-weight support. '
            'Restart the runtime and rerun the dependency cell.'
        ) from exc

    model.eval()
    model.requires_grad_(False)
    parameter_refs = dict(model.named_parameters())
    buffer_refs = dict(model.named_buffers())
    reader = GGUFReader(str(gguf_path))
    quantized_types = {gguf.GGMLQuantizationType.F32, gguf.GGMLQuantizationType.F16}
    loaded = 0
    skipped = []

    print(f'[Streaming T5 load] {gguf_path} -> {device}')
    with torch.inference_mode():
        for tensor in reader.tensors:
            name = tensor.name
            for old_key, new_key in key_map.items():
                name = name.replace(old_key, new_key)
            shape = torch.Size(tuple(int(v) for v in reversed(tensor.shape)))
            raw = torch.from_numpy(np.array(tensor.data))
            is_quantized = tensor.tensor_type not in quantized_types
            if is_quantized:
                quant_param = gguf_loader.GGUFParameter(raw, quant_type=tensor.tensor_type)
                value = gguf_loader.dequantize_gguf_tensor(quant_param, target_dtype=torch.float16)
            else:
                value = raw.to(dtype=torch.float16)
            if value.numel() != math.prod(shape):
                skipped.append((name, 'numel mismatch'))
                del raw, value
                continue
            value = value.reshape(shape)
            target = parameter_refs.get(name)
            if target is None:
                target = buffer_refs.get(name)
            if target is None or tuple(target.shape) != tuple(shape):
                skipped.append((name, 'missing or shape mismatch'))
                del raw, value
                continue
            target.data.copy_(value.to(device=target.device, dtype=target.dtype))
            loaded += 1
            del raw, value

    del reader, parameter_refs, buffer_refs
    gc.collect()
    # The model was materialized on the requested device already.
    # Keep this safety path for unusual device-string inputs.
    if str(next(model.parameters()).device) != str(torch.device(device)):
        model.to(device)
    model.eval()
    model.requires_grad_(False)
    print(f'[Streaming T5 load complete] tensors loaded: {loaded}, skipped: {len(skipped)}')
    if skipped:
        print('First skipped tensors:', skipped[:5])
    return model

print('Memory-efficient T5 loader is ready.')

## Import the unofficial loader

The upstream GGUF script contains a NumPy 2 compatibility assignment to `np.ndarray.newbyteorder`. That assignment can fail on some Colab runtimes because NumPy types are immutable. The next cell creates a temporary sanitized copy of the loader and removes only that obsolete compatibility block.

In [ ]:
sys.path.insert(0, str(PORT_DIR))
sys.path.insert(0, str(OFFICIAL_DIR))

loader_source = PORT_SCRIPT.read_text()
compat_pattern = r"\n    # Apply NumPy 2\.0 compatibility patch.*?\n    # Load GGUF state dict"
loader_source, replacements = re.subn(
    compat_pattern,
    '\n    # NumPy compatibility is handled by the installed gguf package.\n    # Load GGUF state dict',
    loader_source,
    count=1,
    flags=re.S,
)
print('Removed obsolete NumPy compatibility block:', replacements == 1)

PATCHED_LOADER = PORT_DIR / 'generate_image_2b_q8_gguf_colab.py'
PATCHED_LOADER.write_text(loader_source)
spec = importlib.util.spec_from_file_location('infinity_gguf_colab_loader', PATCHED_LOADER)
gguf_loader = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = gguf_loader
spec.loader.exec_module(gguf_loader)
print('Custom GGUF loader imported successfully.')

## Load all components

The T5 encoder is streamed directly to CUDA to reduce Colab host-RAM pressure. The VAE and quantized Infinity transformer are placed on the GPU.

In [ ]:
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('A CUDA GPU is required for practical inference. Select a GPU runtime and rerun.')

print('[1/4] Loading T5 tokenizer...')
text_tokenizer = gguf_loader.load_t5_tokenizer_from_gguf(str(T5_GGUF))

print(f'[2/4] Streaming quantized T5 encoder to {T5_DEVICE}...')
text_encoder = load_t5_encoder_streaming(str(T5_GGUF), device=T5_DEVICE)

print('[3/4] Loading VAE on GPU...')
vae = gguf_loader.load_vae(str(VAE_PATH), vae_type=32, device=DEVICE)

print('[4/4] Loading quantized Infinity-2B transformer on GPU...')
infinity_model = gguf_loader.load_infinity_from_gguf(
    str(INFINITY_GGUF),
    vae=vae,
    device=DEVICE,
    model_type='infinity_2b',
    text_channels=2048,
    pn=MODEL_PN,
)

infinity_model.eval()
vae.eval()
print('All components loaded successfully.')

In [ ]:
# Build the official dynamic-resolution schedule for the selected preset.
import numpy as np
from infinity.utils.dynamic_resolution import dynamic_resolution_h_w, h_div_w_templates

ASPECT_RATIO = 1.0
h_div_w_template = h_div_w_templates[np.argmin(np.abs(h_div_w_templates - ASPECT_RATIO))]
scale_schedule = dynamic_resolution_h_w[h_div_w_template][MODEL_PN]['scales']
scale_schedule = [(1, h, w) for (_, h, w) in scale_schedule]
print('Aspect ratio:', h_div_w_template)
print('Preset:', MODEL_PN)
print('Scale schedule:', scale_schedule)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

def tensor_to_pil(image):
    """Convert common Infinity output layouts/ranges into an RGB PIL image."""
    if isinstance(image, (list, tuple)):
        image = image[0]
    tensor = image.detach().float().cpu() if torch.is_tensor(image) else torch.as_tensor(image).float()
    if tensor.ndim == 4:
        tensor = tensor[0]
    if tensor.ndim != 3:
        raise ValueError(f'Unexpected image shape: {tuple(tensor.shape)}')
    if tensor.shape[0] in (1, 3, 4):
        tensor = tensor.permute(1, 2, 0)
    if tensor.shape[-1] == 1:
        tensor = tensor.repeat(1, 1, 3)
    if tensor.shape[-1] > 3:
        tensor = tensor[..., :3]
    lo, hi = float(tensor.min()), float(tensor.max())
    if lo < -0.05:
        tensor = (tensor + 1.0) / 2.0
    elif hi > 1.05:
        tensor = tensor / 255.0
    array = (tensor.clamp(0, 1).numpy() * 255).round().astype('uint8')
    return Image.fromarray(array, mode='RGB')

def generate_one(prompt, seed=SEED, output_path=None):
    started = time.time()
    with torch.inference_mode():
        image = gguf_loader.generate_image(
            infinity_model, vae, text_tokenizer, text_encoder, prompt,
            cfg_scale=CFG_SCALE,
            tau=TAU,
            seed=seed,
            scale_schedule=scale_schedule,
            vae_type=32,
            device=DEVICE,
        )
    pil = tensor_to_pil(image)
    if output_path is not None:
        pil.save(output_path)
    del image
    gc.collect()
    torch.cuda.empty_cache()
    print(f'Generated in {time.time() - started:.2f}s:', prompt)
    return pil

In [ ]:
import types
from contextlib import nullcontext
import torchvision
import torch.nn.functional as F
from tqdm.auto import tqdm
from PIL import Image, ImageOps
from infinity.models.basic import CrossAttnBlock, apply_rotary_emb, slow_attn
from infinity.models.infinity import sample_with_top_k_top_p_also_inplace_modifying_logits_

RUNTIME_ROOT = Path('/content') if Path('/content').exists() else Path.cwd()
device = DEVICE
infinity = infinity_model
SCALE_SCHEDULE = scale_schedule
PATCH_NUMS = tuple(h for (_, h, w) in SCALE_SCHEDULE)
IMAGE_SIZE_HW = (512, 512)

if not hasattr(vae.quantizer, 'lfq'):
    vae.quantizer.lfq = vae.quantizer.bsq

print('Notebook 13 compatibility aliases ready.')
print('Backend: Infinity-2B GGUF | device:', device, '| image size:', IMAGE_SIZE_HW)

## CSD100 screenshot-example pair configuration

The CSD100 folder is expected to contain directories named like `<object>+<style>`, each with `00.jpg`. This cell uses exactly the 10 example content/style pairs shown in the two screenshots you provided.


In [ ]:
from dataclasses import dataclass
import csv
import itertools
import json
import pandas as pd

WORKSPACE_REPO = 'https://github.com/LeeHoang2710/Style-Transfer-Experiment.git'
WORKSPACE = Path('/Users/builehoang/Documents/Projects/Image Generation/VAR_Style_Transfer_Workspace')
if not WORKSPACE.exists():
    WORKSPACE = Path('/content/VAR_Style_Transfer_Workspace')
    if not WORKSPACE.exists() and Path('/content').exists():
        subprocess.run(['git', 'clone', '--depth', '1', WORKSPACE_REPO, str(WORKSPACE)], check=True)
    if not WORKSPACE.exists():
        raise FileNotFoundError('Could not find VAR_Style_Transfer_Workspace locally or under /content.')

CSD100_DIR = WORKSPACE / 'csd100'
if not CSD100_DIR.exists():
    raise FileNotFoundError(f'Missing CSD100 directory: {CSD100_DIR}')

KINGFISHER_OUTPUT_DIR = WORKSPACE / 'outputs' / 'notebook_13_kingfisher_csd100'
PAIR_OUTPUT_DIR = KINGFISHER_OUTPUT_DIR / 'pairs'
CACHE_DIR = KINGFISHER_OUTPUT_DIR / 'cache'
GALLERY_DIR = KINGFISHER_OUTPUT_DIR / 'galleries'
for path in (PAIR_OUTPUT_DIR, CACHE_DIR, GALLERY_DIR):
    path.mkdir(parents=True, exist_ok=True)

# Paper prompt templates, copied from the method/supplement text.
NEUTRAL_PROMPTS = [
    'a plain realistic photograph with natural colors and ordinary daylight',
    'a standard documentary photograph with neutral balanced lighting',
    'a clean product photograph with a simple background and natural texture',
    'a realistic snapshot with balanced colors and soft natural illumination',
    'a straightforward studio photograph with ordinary materials and neutral tones',
]

OBJECT_PROMPT_TEMPLATES = [
    'a clearly recognizable {object}',
    'a detailed {object}',
    'a centered {object}',
    'a close-up view of a {object}',
    'a {object} with a clear silhouette',
    'a realistic photograph of a {object}',
]


def parse_csd100_item(folder):
    folder = Path(folder)
    if '+' not in folder.name:
        raise ValueError(f'CSD100 folder name must contain +: {folder.name}')
    object_label, style_label = folder.name.split('+', 1)
    clean_object = object_label.replace('_', ' ').replace('-', ' ')
    clean_style = style_label.replace('_', ' ').replace('-', ' ')
    image_path = folder / '00.jpg'
    if not image_path.exists():
        raise FileNotFoundError(image_path)
    return {
        'folder': folder,
        'image_path': image_path,
        'object_label': clean_object,
        'style_label': clean_style,
        'item_id': folder.name,
    }

csd_items = [parse_csd100_item(path) for path in sorted(CSD100_DIR.iterdir()) if path.is_dir() and (path / '00.jpg').exists()]
assert len(csd_items) >= 20, f'Expected at least 20 CSD100 items, found {len(csd_items)}'

# Explicit example pairs from the two provided comparison screenshots.
# Format: (content_source_folder, style_reference_folder).
EXAMPLE_PAIRS = [
    # Screenshot 1
    ('fox+graffiti', 'pen+artwork'),
    ('mushroom+melting_golden_3D_rendering', 'horse+rainbow_flowing_smoke_wave'),
    ('scarecrow+melting_golden_3D_rendering', 'cat+glowing'),
    ('bear+glowing', 'teapot+psychedelic'),
    ('piano+impressionism', 'duck+blueprint'),
    # Screenshot 2
    ('duck+blueprint', 'flower+mosaic'),
    ('flower+pixel', 'brush+watercolor'),
    ('moose+origami', 'cat+glowing'),
    ('horseshoe+graffiti', 'lantern+line_drawing_illustration_art'),
    ('muffin+drawing', 'teacup+comic'),
]

items_by_id = {item['item_id']: item for item in csd_items}
missing_pair_items = sorted({item_id for pair in EXAMPLE_PAIRS for item_id in pair} - set(items_by_id))
if missing_pair_items:
    raise FileNotFoundError('Missing CSD100 example item folders: ' + ', '.join(missing_pair_items))

PAIR_ROWS = []
for pair_id, (content_item_id, style_item_id) in enumerate(EXAMPLE_PAIRS):
    content_item = items_by_id[content_item_id]
    style_item = items_by_id[style_item_id]
    target_prompt = f"a photo of {content_item['object_label']} in {style_item['style_label']} style"
    PAIR_ROWS.append({
        'pair_id': pair_id,
        'content_id': content_item['item_id'],
        'content_object': content_item['object_label'],
        'content_style': content_item['style_label'],
        'content_path': content_item['image_path'],
        'style_id': style_item['item_id'],
        'style_object': style_item['object_label'],
        'style_label': style_item['style_label'],
        'style_path': style_item['image_path'],
        'target_prompt': target_prompt,
    })

pair_table = pd.DataFrame([{k: str(v) for k, v in row.items()} for row in PAIR_ROWS])
display(pair_table[['pair_id', 'content_id', 'content_object', 'content_style', 'style_id', 'style_object', 'style_label', 'target_prompt']])
print('CSD100 items:', len(csd_items))
print('Selected example pairs:', len(PAIR_ROWS))
print('Output directory:', KINGFISHER_OUTPUT_DIR)


## Residual scale-code helpers

Infinity's tokenizer returns bit labels for each residual scale. We convert those labels back into residual code tensors and upsample each residual to the final latent grid, because this is the representation the existing lightweight Infinity generation loop already uses for scale-wise cumulative decoding.

In [ ]:
def _image_tensor_to_pil(image_01):
    tensor = image_01.detach().float().cpu()
    if tensor.ndim == 4:
        tensor = tensor[0]
    array = tensor.clamp(0, 1).permute(1, 2, 0).mul(255).round().byte().numpy()
    return Image.fromarray(array, mode='RGB')


def _save_image_tensor(image_01, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    _image_tensor_to_pil(image_01).save(path)


def load_reference_image(path, size=512):
    image = Image.open(path).convert('RGB')
    image = ImageOps.fit(image, (size, size), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    image_01 = torchvision.transforms.functional.to_tensor(image).unsqueeze(0).to(device)
    return image_01.mul(2).sub(1), image_01, image


def _bit_labels_to_codes(idx_bld, pn):
    idx = idx_bld.reshape(idx_bld.shape[0], pn[1], pn[2], -1)
    idx = idx.unsqueeze(1)
    return vae.quantizer.lfq.indices_to_codes(idx, label_type='bit_label')


def _next_raw_from_summed_codes(summed_codes, next_scale):
    last_stage = F.interpolate(summed_codes, size=next_scale, mode=vae.quantizer.z_interplote_up)
    last_stage = last_stage.squeeze(-3)
    if infinity.apply_spatial_patchify:
        last_stage = torch.nn.functional.pixel_unshuffle(last_stage, 2)
    last_stage = last_stage.reshape(*last_stage.shape[:2], -1).permute(0, 2, 1)
    return last_stage


def _decode_summed_codes_to_image_01(summed_codes):
    image = vae.decode(summed_codes.squeeze(-3))
    return image.add(1).mul(0.5).clamp(0, 1)


@torch.no_grad()
def encode_image_to_residual_codes(image_path):
    image_m11, image_01, pil = load_reference_image(image_path, size=IMAGE_SIZE_HW[0])
    final_size = SCALE_SCHEDULE[-1]
    with torch.amp.autocast('cuda', enabled=False):
        _, _, _, all_bit_indices, _, _ = vae.encode(image_m11.float(), scale_schedule=SCALE_SCHEDULE)

    residuals = []
    native_residuals = []
    for step_id, bit_indices in enumerate(all_bit_indices):
        codes = vae.quantizer.lfq.indices_to_codes(bit_indices, label_type='bit_label').detach().float()
        native_residuals.append(codes.clone())
        if step_id != len(SCALE_SCHEDULE) - 1:
            codes = F.interpolate(codes, size=final_size, mode=vae.quantizer.z_interplote_up)
        residuals.append(codes.detach().float())

    reconstruction = _decode_summed_codes_to_image_01(torch.stack(residuals, dim=0).sum(dim=0))
    return {
        'path': Path(image_path),
        'image_01': image_01.detach().float().cpu(),
        'residuals': residuals,
        'native_residuals': native_residuals,
        'reconstruction_01': reconstruction.detach().float().cpu(),
    }


IMAGE_CODE_CACHE = {}

def get_encoded_image(image_path):
    key = str(Path(image_path))
    if key not in IMAGE_CODE_CACHE:
        IMAGE_CODE_CACHE[key] = encode_image_to_residual_codes(image_path)
    return IMAGE_CODE_CACHE[key]

## Frozen generator queries conditioned on a fixed prefix

This is the key operation from the paper: predict scale `i` under a prompt while keeping the preceding real-image prefix fixed. These predictions build the neutral and content/object subspaces.

In [ ]:
def encode_prompts(prompts):
    if isinstance(prompts, str):
        prompts = [prompts]
    tokens = text_tokenizer(
        text=list(prompts), max_length=512, padding='max_length', truncation=True, return_tensors='pt'
    )
    input_ids = tokens.input_ids.to(device, non_blocking=True)
    mask = tokens.attention_mask.to(device, non_blocking=True)
    with torch.no_grad():
        text_features = text_encoder(input_ids=input_ids, attention_mask=mask)['last_hidden_state'].float()
    lens = mask.sum(dim=-1).tolist()
    cu_seqlens_k = F.pad(mask.sum(dim=-1).to(dtype=torch.int32).cumsum_(0), (1, 0))
    max_seqlen_k = max(lens)
    kv_compact = []
    for len_i, feat_i in zip(lens, text_features.unbind(0)):
        kv_compact.append(feat_i[:len_i])
    kv_compact = torch.cat(kv_compact, dim=0)
    return kv_compact, lens, cu_seqlens_k, max_seqlen_k


def _sample_or_argmax_bit_labels(logits_bl2d, rng=None, top_k=600, top_p=0.95, mode='argmax'):
    batch, seq_len = logits_bl2d.shape[:2]
    logits = logits_bl2d.reshape(batch, -1, 2).clone()
    if mode == 'argmax':
        sampled = logits.argmax(dim=-1)
    elif mode == 'sample':
        sampled = sample_with_top_k_top_p_also_inplace_modifying_logits_(
            logits, rng=rng, top_k=top_k, top_p=top_p, num_samples=1
        )[:, :, 0]
    else:
        raise ValueError(f'Unknown prediction mode: {mode}')
    return sampled.reshape(batch, seq_len, -1)


@torch.no_grad()
def predict_scale_code_from_prefix(
    prefix_residuals,
    prompt,
    target_step,
    *,
    cfg=1.0,
    tau=0.1,
    top_k=600,
    top_p=0.95,
    seed=42,
    mode='argmax',
):
    """Predict residual code R_i(prompt) with R_<i fixed to prefix_residuals.

    prefix_residuals are final-grid residual code tensors from encode_image_to_residual_codes.
    The returned tensor is also upsampled to the final latent grid, matching Ri.
    """
    model = infinity
    model.eval()
    bs = 2  # conditional + unconditional for CFG
    rng = torch.Generator(device=device).manual_seed(seed)

    kv_compact, lens, cu_seqlens_k, max_seqlen_k = encode_prompts([prompt])
    kv_compact_un = kv_compact.clone()
    kv_compact_un[:lens[0]] = model.cfg_uncond[:lens[0]]
    kv_compact = torch.cat((kv_compact, kv_compact_un), dim=0)
    cu_seqlens_k = torch.cat((cu_seqlens_k, cu_seqlens_k[1:] + cu_seqlens_k[-1]), dim=0)

    kv_compact = model.text_norm(kv_compact)
    cond_BD = model.text_proj_for_sos((kv_compact, cu_seqlens_k, max_seqlen_k))
    kv_compact = model.text_proj_for_ca(kv_compact)
    ca_kv = kv_compact, cu_seqlens_k, max_seqlen_k
    last_stage = cond_BD.unsqueeze(1).expand(bs, 1, -1) + model.pos_start.expand(bs, 1, -1)

    with torch.amp.autocast('cuda', enabled=False):
        cond_BD_or_gss = model.shared_ada_lin(cond_BD.float()).float().contiguous()

    final_size = SCALE_SCHEDULE[-1]
    summed_codes = last_stage.new_zeros(1, model.d_vae, *final_size)

    for block in model.unregistered_blocks:
        block.sa.kv_caching(True)

    try:
        with torch.amp.autocast('cuda', enabled=True, dtype=torch.bfloat16, cache_enabled=True):
            for step_id, pn in enumerate(SCALE_SCHEDULE):
                need_to_pad = 0
                attn_fn = None
                if model.use_flex_attn:
                    attn_fn = model.attn_fn_compile_dict.get(tuple(SCALE_SCHEDULE[:step_id + 1]), None)

                for block_idx, block_chunk in enumerate(model.block_chunks):
                    if model.add_lvl_embeding_only_first_block and block_idx == 0:
                        last_stage = model.add_lvl_embeding(last_stage, step_id, SCALE_SCHEDULE, need_to_pad=need_to_pad)
                    if not model.add_lvl_embeding_only_first_block:
                        last_stage = model.add_lvl_embeding(last_stage, step_id, SCALE_SCHEDULE, need_to_pad=need_to_pad)
                    for block in block_chunk.module:
                        last_stage = block(
                            x=last_stage,
                            cond_BD=cond_BD_or_gss,
                            ca_kv=ca_kv,
                            attn_bias_or_two_vector=None,
                            attn_fn=attn_fn,
                            scale_schedule=SCALE_SCHEDULE,
                            rope2d_freqs_grid=model.rope2d_freqs_grid,
                            scale_ind=step_id,
                        )

                logits = model.get_logits(last_stage, cond_BD).mul(1 / float(tau))
                logits = float(cfg) * logits[:1] + (1 - float(cfg)) * logits[1:]

                if step_id == target_step:
                    idx = _sample_or_argmax_bit_labels(logits, rng, top_k, top_p, mode=mode)
                    codes = _bit_labels_to_codes(idx, pn).detach().float()
                    if step_id != len(SCALE_SCHEDULE) - 1:
                        codes = F.interpolate(codes, size=final_size, mode=vae.quantizer.z_interplote_up)
                    return codes.detach().float()

                # Keep the real prefix fixed for all steps before target_step.
                if step_id >= len(prefix_residuals):
                    raise ValueError('prefix_residuals ended before target_step.')
                summed_codes = summed_codes + prefix_residuals[step_id].to(device=device, dtype=summed_codes.dtype)
                next_scale = SCALE_SCHEDULE[step_id + 1]
                next_raw = _next_raw_from_summed_codes(summed_codes, next_scale)
                last_stage = model.word_embed(model.norm0_ve(next_raw)).repeat(bs, 1, 1)
    finally:
        for block in model.unregistered_blocks:
            block.sa.kv_caching(False)

    raise RuntimeError('target_step was not reached.')

## Subspace construction and projection

`N_i = mu_i^N + span(V_i^N)` is represented by its mean tensor and top SVD basis rows. The content/object subspace is linear and built from object-prompt directions after subtracting the neutral mean.

In [ ]:
@dataclass
class AffineSubspace:
    mean: torch.Tensor
    basis: torch.Tensor  # [rank, flattened_dim], orthonormal rows

@dataclass
class LinearSubspace:
    basis: torch.Tensor  # [rank, flattened_dim], orthonormal rows


def _flatten(x):
    return x.detach().float().reshape(-1)


def _fit_basis_from_rows(rows, rank):
    # rows: [num_samples, flattened_dim]
    rows = rows.float()
    max_rank = min(int(rank), rows.shape[0], rows.shape[1])
    if max_rank < 1:
        raise ValueError('rank must be at least 1 and compatible with sample matrix.')
    _, _, vh = torch.linalg.svd(rows, full_matrices=False)
    return vh[:max_rank].contiguous()


def project_linear(x, subspace):
    shape = x.shape
    x_flat = x.float().reshape(-1)
    basis = subspace.basis.to(x_flat.device, x_flat.dtype)
    coeff = x_flat @ basis.T
    projected = coeff @ basis
    return projected.reshape(shape)


def project_affine(x, subspace):
    shape = x.shape
    mean = subspace.mean.to(x.device, x.dtype)
    basis = subspace.basis.to(x.device, x.dtype)
    diff_flat = (x.float() - mean.float()).reshape(-1)
    coeff = diff_flat @ basis.T
    projected = mean.float().reshape(-1) + coeff @ basis
    return projected.reshape(shape).to(dtype=x.dtype)


SUBSPACE_CACHE = {}


def build_neutral_subspace(encoded, scale_index, *, rank=4, prediction_mode='argmax'):
    key = ('neutral', str(encoded['path']), int(scale_index), int(rank), prediction_mode)
    if key in SUBSPACE_CACHE:
        return SUBSPACE_CACHE[key]

    predictions = []
    for prompt_id, prompt in enumerate(NEUTRAL_PROMPTS):
        pred = predict_scale_code_from_prefix(
            encoded['residuals'], prompt, scale_index,
            cfg=1.0, tau=TAU, top_k=600, top_p=0.95,
            seed=SEED + 1000 * scale_index + prompt_id,
            mode=prediction_mode,
        )
        predictions.append(pred.detach().float())
    stack = torch.stack(predictions, dim=0).squeeze(1)  # [Q, C, 1, H, W]
    mean = stack.mean(dim=0, keepdim=True)
    centered = (stack - mean).reshape(stack.shape[0], -1)
    basis = _fit_basis_from_rows(centered, rank=rank)
    result = AffineSubspace(mean=mean.detach().float(), basis=basis.detach().float())
    SUBSPACE_CACHE[key] = result
    return result


def build_content_subspace(encoded, scale_index, object_label, neutral_subspace, *, rank=4, prediction_mode='argmax'):
    key = ('content', str(encoded['path']), str(object_label), int(scale_index), int(rank), prediction_mode)
    if key in SUBSPACE_CACHE:
        return SUBSPACE_CACHE[key]

    object_prompts = [template.format(object=object_label) for template in OBJECT_PROMPT_TEMPLATES]
    directions = []
    for prompt_id, prompt in enumerate(object_prompts):
        pred = predict_scale_code_from_prefix(
            encoded['residuals'], prompt, scale_index,
            cfg=1.0, tau=TAU, top_k=600, top_p=0.95,
            seed=SEED + 2000 * scale_index + prompt_id,
            mode=prediction_mode,
        )
        directions.append((pred.detach().float() - neutral_subspace.mean.detach().float()).reshape(-1))
    matrix = torch.stack(directions, dim=0)
    basis = _fit_basis_from_rows(matrix, rank=rank)
    result = LinearSubspace(basis=basis.detach().float())
    SUBSPACE_CACHE[key] = result
    return result

## Five-loss decomposition optimization

For each fixed real scale code `R_i`, we optimize only `C_i` and `S_i`. The encoded `R_i`, neutral subspace, and content subspace stay frozen.

In [ ]:
SELECTED_SCALE_INDICES = [0, 1, 3]  # paper R1, R2, R4 in zero-based code
PRESERVED_SCALE_INDEX = 2           # paper R3
SUBSPACE_RANK_NEUTRAL = 4
SUBSPACE_RANK_CONTENT = 4
DECOMP_STEPS = 250
DECOMP_LR = 3e-2
BETA_R4 = 0.5
LOSS_WEIGHTS = {
    'sty': 1.0,
    'align': 0.5,
    'remove': 0.75,
    'rec': 0.05,
    'orth': 0.25,
}
# Mean-normalized squared norm is numerically stable for very large latent grids.
# Set to 'sum' if you want the literal unnormalized ||.||_2^2.
LOSS_REDUCTION = 'mean'
PREDICTION_MODE_FOR_SUBSPACES = 'argmax'


def squared_norm(x):
    x = x.float()
    if LOSS_REDUCTION == 'sum':
        return x.pow(2).sum()
    if LOSS_REDUCTION == 'mean':
        return x.pow(2).mean()
    raise ValueError(f'Unknown LOSS_REDUCTION: {LOSS_REDUCTION}')


def cosine_squared(a, b):
    af = a.float().reshape(1, -1)
    bf = b.float().reshape(1, -1)
    return F.cosine_similarity(af, bf, dim=-1, eps=1e-8).pow(2).mean()


@dataclass
class ScaleDecomposition:
    C: torch.Tensor
    S: torch.Tensor
    neutral: AffineSubspace
    content: LinearSubspace
    losses: list


def initialize_components(Ri, neutral_subspace, content_subspace):
    neutral_projection = project_affine(Ri, neutral_subspace)
    S0 = (Ri - neutral_projection).detach().float()
    C0 = project_linear(Ri - neutral_subspace.mean.to(Ri), content_subspace).detach().float()
    return C0, S0


def decompose_one_scale(Ri, neutral_subspace, content_subspace, *, steps=DECOMP_STEPS, lr=DECOMP_LR):
    Ri = Ri.detach().float().to(device)
    neutral = AffineSubspace(
        mean=neutral_subspace.mean.detach().float().to(device),
        basis=neutral_subspace.basis.detach().float().to(device),
    )
    content = LinearSubspace(basis=content_subspace.basis.detach().float().to(device))

    C0, S0 = initialize_components(Ri, neutral, content)
    C = torch.nn.Parameter(C0.clone())
    S = torch.nn.Parameter(S0.clone())
    optimizer = torch.optim.Adam([C, S], lr=lr)
    history = []

    for step in range(int(steps)):
        optimizer.zero_grad(set_to_none=True)
        style_removed = Ri - S
        Lsty = squared_norm(style_removed - project_affine(style_removed, neutral))
        Lalign = squared_norm(C - project_linear(C, content))
        Lremove = squared_norm(project_linear((Ri - neutral.mean) - C, content))
        Lrec = squared_norm(Ri - (C + S))
        Lorth = cosine_squared(C, S)
        loss = (
            LOSS_WEIGHTS['sty'] * Lsty
            + LOSS_WEIGHTS['align'] * Lalign
            + LOSS_WEIGHTS['remove'] * Lremove
            + LOSS_WEIGHTS['rec'] * Lrec
            + LOSS_WEIGHTS['orth'] * Lorth
        )
        loss.backward()
        optimizer.step()

        if step == 0 or step == steps - 1 or (step + 1) % 50 == 0:
            history.append({
                'step': step + 1,
                'total': float(loss.detach().cpu()),
                'Lsty': float(Lsty.detach().cpu()),
                'Lalign': float(Lalign.detach().cpu()),
                'Lremove': float(Lremove.detach().cpu()),
                'Lrec': float(Lrec.detach().cpu()),
                'Lorth': float(Lorth.detach().cpu()),
            })

    return ScaleDecomposition(
        C=C.detach().float().cpu(),
        S=S.detach().float().cpu(),
        neutral=neutral_subspace,
        content=content_subspace,
        losses=history,
    )


DECOMPOSITION_CACHE = {}


def decompose_image_for_object(image_path, object_label, *, selected_scales=SELECTED_SCALE_INDICES):
    """Compute C_i and S_i for one image under one target object label."""
    image_path = Path(image_path)
    key = (str(image_path), str(object_label), tuple(selected_scales), DECOMP_STEPS, DECOMP_LR, LOSS_REDUCTION)
    if key in DECOMPOSITION_CACHE:
        return DECOMPOSITION_CACHE[key]

    encoded = get_encoded_image(image_path)
    decompositions = {}
    for scale_index in tqdm(selected_scales, desc=f'decompose {image_path.name} as {object_label}', leave=False):
        neutral = build_neutral_subspace(
            encoded, scale_index, rank=SUBSPACE_RANK_NEUTRAL,
            prediction_mode=PREDICTION_MODE_FOR_SUBSPACES,
        )
        content = build_content_subspace(
            encoded, scale_index, object_label, neutral,
            rank=SUBSPACE_RANK_CONTENT,
            prediction_mode=PREDICTION_MODE_FOR_SUBSPACES,
        )
        Ri = encoded['residuals'][scale_index]
        decompositions[scale_index] = decompose_one_scale(Ri, neutral, content)
        gc.collect()
        torch.cuda.empty_cache()

    DECOMPOSITION_CACHE[key] = decompositions
    return decompositions

## Prefix composition and suffix generation

After decomposition, the optimized components are used in the paper prefix:

`R1* = A1 - S_A1 + S_B1`

`R2* = A2 - S_A2 + S_B2`

`R3* = A3`

`R4* = B4 + beta(C_A4 - C_B4)`

Then Infinity predicts `R5...RN` with all model weights frozen.

In [ ]:
@torch.no_grad()
def generate_from_fixed_prefix(
    prefix_residuals,
    prompt,
    *,
    seed=SEED,
    cfg=1.0,
    tau=0.1,
    top_k=600,
    top_p=0.95,
):
    model = infinity
    model.eval()
    bs = 2
    rng = torch.Generator(device=device).manual_seed(seed)

    kv_compact, lens, cu_seqlens_k, max_seqlen_k = encode_prompts([prompt])
    kv_compact_un = kv_compact.clone()
    kv_compact_un[:lens[0]] = model.cfg_uncond[:lens[0]]
    kv_compact = torch.cat((kv_compact, kv_compact_un), dim=0)
    cu_seqlens_k = torch.cat((cu_seqlens_k, cu_seqlens_k[1:] + cu_seqlens_k[-1]), dim=0)

    kv_compact = model.text_norm(kv_compact)
    cond_BD = model.text_proj_for_sos((kv_compact, cu_seqlens_k, max_seqlen_k))
    kv_compact = model.text_proj_for_ca(kv_compact)
    ca_kv = kv_compact, cu_seqlens_k, max_seqlen_k
    last_stage = cond_BD.unsqueeze(1).expand(bs, 1, -1) + model.pos_start.expand(bs, 1, -1)

    with torch.amp.autocast('cuda', enabled=False):
        cond_BD_or_gss = model.shared_ada_lin(cond_BD.float()).float().contiguous()

    final_size = SCALE_SCHEDULE[-1]
    summed_codes = last_stage.new_zeros(1, model.d_vae, *final_size)
    trace = []

    for block in model.unregistered_blocks:
        block.sa.kv_caching(True)

    try:
        with torch.amp.autocast('cuda', enabled=True, dtype=torch.bfloat16, cache_enabled=True):
            for step_id, pn in enumerate(SCALE_SCHEDULE):
                need_to_pad = 0
                attn_fn = None
                if model.use_flex_attn:
                    attn_fn = model.attn_fn_compile_dict.get(tuple(SCALE_SCHEDULE[:step_id + 1]), None)

                for block_idx, block_chunk in enumerate(model.block_chunks):
                    if model.add_lvl_embeding_only_first_block and block_idx == 0:
                        last_stage = model.add_lvl_embeding(last_stage, step_id, SCALE_SCHEDULE, need_to_pad=need_to_pad)
                    if not model.add_lvl_embeding_only_first_block:
                        last_stage = model.add_lvl_embeding(last_stage, step_id, SCALE_SCHEDULE, need_to_pad=need_to_pad)
                    for block in block_chunk.module:
                        last_stage = block(
                            x=last_stage,
                            cond_BD=cond_BD_or_gss,
                            ca_kv=ca_kv,
                            attn_bias_or_two_vector=None,
                            attn_fn=attn_fn,
                            scale_schedule=SCALE_SCHEDULE,
                            rope2d_freqs_grid=model.rope2d_freqs_grid,
                            scale_ind=step_id,
                        )

                if step_id < len(prefix_residuals):
                    codes = prefix_residuals[step_id].to(device=device, dtype=summed_codes.dtype)
                else:
                    logits = model.get_logits(last_stage, cond_BD).mul(1 / float(tau))
                    logits = float(cfg) * logits[:1] + (1 - float(cfg)) * logits[1:]
                    idx = _sample_or_argmax_bit_labels(logits, rng, top_k, top_p, mode='sample')
                    codes = _bit_labels_to_codes(idx, pn).detach().float()
                    if step_id != len(SCALE_SCHEDULE) - 1:
                        codes = F.interpolate(codes, size=final_size, mode=vae.quantizer.z_interplote_up)

                summed_codes = summed_codes + codes
                trace.append(summed_codes.detach().float().cpu())

                if step_id != len(SCALE_SCHEDULE) - 1:
                    next_scale = SCALE_SCHEDULE[step_id + 1]
                    next_raw = _next_raw_from_summed_codes(summed_codes, next_scale)
                    last_stage = model.word_embed(model.norm0_ve(next_raw)).repeat(bs, 1, 1)

        image = _decode_summed_codes_to_image_01(summed_codes)
        return {'image_01': image.detach().float().cpu(), 'trace': trace}
    finally:
        for block in model.unregistered_blocks:
            block.sa.kv_caching(False)


def compose_kingfisher_prefix(content_encoded, style_encoded, content_decomp, style_decomp, beta=BETA_R4):
    A = content_encoded['residuals']
    B = style_encoded['residuals']
    SA1 = content_decomp[0].S.to(A[0])
    SB1 = style_decomp[0].S.to(A[0])
    SA2 = content_decomp[1].S.to(A[1])
    SB2 = style_decomp[1].S.to(A[1])
    CA4 = content_decomp[3].C.to(A[3])
    CB4 = style_decomp[3].C.to(A[3])

    R1 = A[0].detach().float().cpu() - SA1.cpu() + SB1.cpu()
    R2 = A[1].detach().float().cpu() - SA2.cpu() + SB2.cpu()
    R3 = A[2].detach().float().cpu()
    R4 = B[3].detach().float().cpu() + float(beta) * (CA4.cpu() - CB4.cpu())
    return [R1, R2, R3, R4]

## Inspect decomposition losses

Run this before the 30-pair generation if you want to verify that decomposition optimization is behaving correctly. The default helper inspects paper scales `R1`, `R2`, and `R4` together, because those are the optimized scales used by the prefix composition.


In [ ]:
SCALE_LABELS = {0: 'R1', 1: 'R2', 2: 'R3', 3: 'R4'}


def loss_history_dataframe(image_path, object_label, scale_index):
    decomp = decompose_image_for_object(image_path, object_label)
    history = decomp[scale_index].losses
    table = pd.DataFrame(history)
    table.insert(0, 'scale', SCALE_LABELS.get(scale_index, f'R{scale_index + 1}'))
    return table


def inspect_decomposition_losses(
    image_path,
    object_label,
    scale_indices=(0, 1, 3),
    show_tables=True,
    show_plot=True,
    log_y=True,
):
    """
    Plot the optimization trace for one image/object decomposition.

    The default scales are the paper scales R1, R2, and R4. The 3x2 grid shows
    total loss plus all five component losses. Each subplot has one line per scale.
    """
    tables = []
    for scale_index in scale_indices:
        tables.append(loss_history_dataframe(image_path, object_label, scale_index))
    combined = pd.concat(tables, ignore_index=True)

    if show_tables:
        display(combined)

    if show_plot:
        loss_specs = [
            ('total', 'Total weighted loss'),
            ('Lsty', 'Style-removal loss'),
            ('Lalign', 'Content-align loss'),
            ('Lremove', 'Content-removal loss'),
            ('Lrec', 'Reconstruction loss'),
            ('Lorth', 'Orthogonality loss'),
        ]

        figure, axes = plt.subplots(3, 2, figsize=(13, 10), squeeze=False)
        axes_flat = axes.reshape(-1)

        for axis, (loss_key, title) in zip(axes_flat, loss_specs):
            for scale_label, group in combined.groupby('scale', sort=False):
                axis.plot(
                    group['step'],
                    group[loss_key].clip(lower=1e-12),
                    marker='o',
                    linewidth=1.8,
                    label=scale_label,
                )

            axis.set_title(title, fontsize=11)
            axis.set_xlabel('optimization step')
            axis.set_ylabel('loss')
            axis.grid(alpha=0.3)

            if log_y:
                axis.set_yscale('log')

            axis.legend(title='scale', fontsize=8)

        figure.suptitle(
            f"Decomposition optimization losses: {Path(image_path).parent.name} as {object_label}",
            fontsize=14,
        )
        figure.tight_layout()
        plt.show()

    return combined


# Default preview: first content image, automatically showing paper scales R1, R2, and R4.
# This runs decomposition for the first image if it is not already cached.
first = PAIR_ROWS[0]
inspect_decomposition_losses(first['content_path'], first['content_object'], scale_indices=(0, 1, 3))


## Generate the 10 CSD100 pairs

This is the main run cell. It decomposes both images in each selected pair, composes the Kingfisher prefix, and writes one generated image per pair.


In [ ]:
RUN_FULL_10 = True
FORCE_REGENERATE = False
GEN_CFG = 1.0
GEN_TAU = 0.1
GEN_TOP_K = 600
GEN_TOP_P = 0.95

GENERATED_TRACE_DIR = KINGFISHER_OUTPUT_DIR / 'generated_traces'
GENERATED_SCALE_IMAGE_DIR = KINGFISHER_OUTPUT_DIR / 'generated_scale_images'
for path in (GENERATED_TRACE_DIR, GENERATED_SCALE_IMAGE_DIR):
    path.mkdir(parents=True, exist_ok=True)


def pair_slug(row):
    return f"{int(row['pair_id']):02d}_{row['content_id']}__STYLE__{row['style_id']}"


def output_path_for_pair(row):
    return PAIR_OUTPUT_DIR / f"{pair_slug(row)}.png"


def generated_trace_path_for_pair(row):
    return GENERATED_TRACE_DIR / f"{pair_slug(row)}_trace.pt"


def generated_trace_meta_path_for_pair(row):
    return GENERATED_TRACE_DIR / f"{pair_slug(row)}_meta.json"


def generated_scale_dir_for_pair(row):
    return GENERATED_SCALE_IMAGE_DIR / pair_slug(row)


@torch.no_grad()
def save_generated_trace_artifacts(row, result):
    trace = [cumulative_codes.detach().float().cpu() for cumulative_codes in result['trace']]
    trace_path = generated_trace_path_for_pair(row)
    trace_path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(trace, trace_path)

    scale_dir = generated_scale_dir_for_pair(row)
    scale_dir.mkdir(parents=True, exist_ok=True)
    for scale_id, cumulative_codes in enumerate(trace, start=1):
        image_01 = _decode_summed_codes_to_image_01(cumulative_codes.to(device))
        _save_image_tensor(image_01.detach().float().cpu(), scale_dir / f'scale_{scale_id:02d}.png')

    metadata = {
        'pair_id': int(row['pair_id']),
        'content_id': row['content_id'],
        'style_id': row['style_id'],
        'target_prompt': row['target_prompt'],
        'beta_r4': float(BETA_R4),
        'seed': int(SEED + int(row['pair_id'])),
        'cfg': float(GEN_CFG),
        'tau': float(GEN_TAU),
        'top_k': int(GEN_TOP_K),
        'top_p': float(GEN_TOP_P),
        'num_scales': len(trace),
    }
    generated_trace_meta_path_for_pair(row).write_text(json.dumps(metadata, indent=2))
    return trace_path


def generated_artifacts_exist(row):
    return (
        output_path_for_pair(row).exists()
        and generated_trace_path_for_pair(row).exists()
        and generated_trace_meta_path_for_pair(row).exists()
    )


def run_one_pair(row):
    output_path = output_path_for_pair(row)
    if generated_artifacts_exist(row) and not FORCE_REGENERATE:
        return {
            'pair_id': row['pair_id'],
            'output_path': output_path,
            'trace_path': generated_trace_path_for_pair(row),
            'status': 'cached',
        }

    if output_path.exists() and not generated_trace_path_for_pair(row).exists() and not FORCE_REGENERATE:
        print('Final image exists but trace is missing; regenerating trace artifacts for:', pair_slug(row))

    content_encoded = get_encoded_image(row['content_path'])
    style_encoded = get_encoded_image(row['style_path'])

    # Paper-style decomposition: each real image uses object prompts for its own object category.
    content_decomp = decompose_image_for_object(row['content_path'], row['content_object'])
    style_decomp = decompose_image_for_object(row['style_path'], row['style_object'])

    prefix = compose_kingfisher_prefix(content_encoded, style_encoded, content_decomp, style_decomp, beta=BETA_R4)
    result = generate_from_fixed_prefix(
        prefix,
        row['target_prompt'],
        seed=SEED + int(row['pair_id']),
        cfg=GEN_CFG,
        tau=GEN_TAU,
        top_k=GEN_TOP_K,
        top_p=GEN_TOP_P,
    )
    _save_image_tensor(result['image_01'], output_path)
    trace_path = save_generated_trace_artifacts(row, result)
    return {
        'pair_id': row['pair_id'],
        'output_path': output_path,
        'trace_path': trace_path,
        'status': 'generated',
    }


run_records = []
if RUN_FULL_10:
    for row in tqdm(PAIR_ROWS, desc='Kingfisher CSD100 10-pair generation'):
        record = run_one_pair(row)
        run_records.append(record)
        print(record['status'], record['pair_id'], record['output_path'].name)
        gc.collect()
        torch.cuda.empty_cache()

    records_df = pd.DataFrame([{k: str(v) for k, v in record.items()} for record in run_records])
    records_df.to_csv(KINGFISHER_OUTPUT_DIR / 'generation_records.csv', index=False)
    display(records_df)
else:
    print('RUN_FULL_10 is False; set it to True to generate all 10 pairs.')


## Scale reconstruction diagnostics

This section plots one chart per pair. Each chart decodes the cumulative image after every scale for the content source, style reference, and generated Kingfisher trajectory. Use it to locate the scale where content begins to drift or style-reference structure leaks in.


In [ ]:
DIAGNOSTIC_DIR = KINGFISHER_OUTPUT_DIR / 'scale_diagnostics'
DIAGNOSTIC_DIR.mkdir(parents=True, exist_ok=True)


def cumulative_trace_from_residuals(residuals):
    cumulative = None
    trace = []
    for residual in residuals:
        current = residual.detach().float().to(device)
        cumulative = current if cumulative is None else cumulative + current
        trace.append(cumulative.detach().float().cpu())
    return trace


@torch.no_grad()
def decode_trace_to_pil_images(trace):
    images = []
    for cumulative_codes in trace:
        image_01 = _decode_summed_codes_to_image_01(cumulative_codes.to(device))
        images.append(_image_tensor_to_pil(image_01))
    return images


def load_generated_trace_for_pair(row, regenerate_if_missing=True):
    trace_path = generated_trace_path_for_pair(row)
    if not trace_path.exists():
        if not regenerate_if_missing:
            raise FileNotFoundError(f'Missing generated trace: {trace_path}')
        print('Generated trace missing; running generation once for:', pair_slug(row))
        run_one_pair(row)
    return torch.load(trace_path, map_location='cpu')


def plot_scale_reconstruction_chart(row, save=True):
    content_encoded = get_encoded_image(row['content_path'])
    style_encoded = get_encoded_image(row['style_path'])

    content_trace = cumulative_trace_from_residuals(content_encoded['residuals'])
    style_trace = cumulative_trace_from_residuals(style_encoded['residuals'])
    generated_trace = load_generated_trace_for_pair(row)

    traces = [content_trace, style_trace, generated_trace]
    row_labels = [
        f"Content A\n{row['content_object']}\n({row['content_style']})",
        f"Style B\n{row['style_object']}\n({row['style_label']})",
        f"Generated\n{row['content_object']} in\n{row['style_label']} style",
    ]

    decoded_rows = [decode_trace_to_pil_images(trace) for trace in traces]
    num_scales = len(decoded_rows[0])
    fig, axes = plt.subplots(3, num_scales + 1, figsize=(1.55 * (num_scales + 1), 5.2), squeeze=False)

    for axis in axes.reshape(-1):
        axis.axis('off')

    for row_id, (label, images) in enumerate(zip(row_labels, decoded_rows)):
        axes[row_id, 0].text(0.5, 0.5, label, ha='center', va='center', fontsize=9, wrap=True)
        for scale_id, image in enumerate(images):
            axes[row_id, scale_id + 1].imshow(image)
            if row_id == 0:
                axes[row_id, scale_id + 1].set_title(f'R{scale_id + 1}', fontsize=8)

    fig.suptitle(
        f"Pair {int(row['pair_id']):02d}: {row['content_id']}  ->  {row['style_id']} | beta R4={BETA_R4}",
        fontsize=11,
    )
    fig.tight_layout()

    if save:
        save_path = DIAGNOSTIC_DIR / f"{pair_slug(row)}_scale_recon.png"
        fig.savefig(save_path, dpi=180, bbox_inches='tight')
        print('Saved diagnostic:', save_path)

    plt.show()
    plt.close(fig)


def plot_all_scale_reconstruction_charts(pair_rows=PAIR_ROWS):
    for row in tqdm(pair_rows, desc='Scale reconstruction diagnostics'):
        plot_scale_reconstruction_chart(row, save=True)
        gc.collect()
        torch.cuda.empty_cache()


RUN_SCALE_RECON_DIAGNOSTICS = True
if RUN_SCALE_RECON_DIAGNOSTICS:
    plot_all_scale_reconstruction_charts(PAIR_ROWS)
else:
    print('Set RUN_SCALE_RECON_DIAGNOSTICS = True to generate the 10 scale reconstruction charts.')


## Gallery

This cell creates a compact 10-row gallery with content source, style reference, and generated result.


In [ ]:
def show_kingfisher_gallery():
    rows = []
    for row in PAIR_ROWS:
        output_path = PAIR_OUTPUT_DIR / f"{int(row['pair_id']):02d}_{row['content_id']}__STYLE__{row['style_id']}.png"
        if output_path.exists():
            rows.append((row, output_path))
    if not rows:
        print('No generated images found yet.')
        return

    fig, axes = plt.subplots(len(rows), 3, figsize=(10.5, 3.4 * len(rows)), squeeze=False)
    column_titles = ['content source', 'style reference', 'generated output']

    for row_id, (row, output_path) in enumerate(rows):
        content_image = Image.open(row['content_path']).convert('RGB')
        style_image = Image.open(row['style_path']).convert('RGB')
        output_image = Image.open(output_path).convert('RGB')
        panels = [content_image, style_image, output_image]
        titles = [
            f"content: {row['content_object']}\nsource style: {row['content_style']}",
            f"style: {row['style_label']}\nobject: {row['style_object']}",
            f"generated\n{row['target_prompt']}",
        ]

        for col_id, (image, title) in enumerate(zip(panels, titles)):
            axes[row_id, col_id].imshow(image)
            if row_id == 0:
                axes[row_id, col_id].set_title(f"{column_titles[col_id]}\n{title}", fontsize=9)
            else:
                axes[row_id, col_id].set_title(title, fontsize=9)
            axes[row_id, col_id].axis('off')

    fig.suptitle('Notebook 13 Kingfisher CSD100 10-pair outputs', fontsize=14)
    fig.tight_layout()
    gallery_path = GALLERY_DIR / 'kingfisher_csd100_10_pair_gallery.png'
    fig.savefig(gallery_path, dpi=180, bbox_inches='tight')
    print('Saved gallery:', gallery_path)
    plt.show()

show_kingfisher_gallery()


## Download gallery images

Run this after generating the 10 pairs and gallery. In Colab it starts a browser download; locally it writes a zip file and prints the path.


In [ ]:
import zipfile

def package_kingfisher_outputs_for_download(include_pair_images=True, include_gallery=True, include_records=True):
    zip_path = KINGFISHER_OUTPUT_DIR / 'notebook_13_kingfisher_csd100_outputs.zip'
    files_to_add = []

    if include_pair_images and PAIR_OUTPUT_DIR.exists():
        files_to_add.extend(sorted(PAIR_OUTPUT_DIR.glob('*.png')))

    if include_gallery and GALLERY_DIR.exists():
        files_to_add.extend(sorted(GALLERY_DIR.glob('*.png')))

    if include_records:
        for extra_path in [KINGFISHER_OUTPUT_DIR / 'generation_records.csv']:
            if extra_path.exists():
                files_to_add.append(extra_path)

    if not files_to_add:
        raise FileNotFoundError('No generated images or gallery files found yet. Run the generation and gallery cells first.')

    with zipfile.ZipFile(zip_path, mode='w', compression=zipfile.ZIP_DEFLATED) as archive:
        for file_path in files_to_add:
            archive.write(file_path, arcname=file_path.relative_to(KINGFISHER_OUTPUT_DIR))

    print(f'Packaged {len(files_to_add)} files: {zip_path}')

    try:
        from google.colab import files
        files.download(str(zip_path))
    except Exception:
        print('Download helper is only automatic in Colab. Local zip path:')
        print(zip_path)

    return zip_path

package_kingfisher_outputs_for_download()